# Customer Segmentation Analysis Using Python

This notebook analyses the Mall Customers dataset to identify customer groups based on annual income and spending score. The project includes data inspection, data quality checks, exploratory data analysis, correlation analysis, and K-Means clustering.

In [ ]:
# Importing required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Creating a folder to save charts

import os
os.makedirs("images", exist_ok=True)

## 1. Data Loading

The dataset is loaded as a pandas DataFrame. The file path below works when the dataset is stored in the same folder as the notebook. If your dataset is saved in a separate `dataset` folder on GitHub, use the alternative path shown in the comment.

In [ ]:
# Loading the dataset

file_path = "Mall_Customers.csv"

# If using GitHub folder structure, use this instead:
# file_path = "../dataset/Mall_Customers.csv"

df = pd.read_csv(file_path)

# Displaying first five records
df.head()

## 2. Data Understanding

The following cells inspect the dataset structure, dimensions, data types, and summary statistics.

In [ ]:
# Checking dataset dimensions

df.shape

In [ ]:
# Checking column names

df.columns

In [ ]:
# Checking dataset information

df.info()

In [ ]:
# Summary statistics for numerical variables

df.describe()

## 3. Data Quality Assessment

The dataset is checked for missing values and duplicate records before analysis.

In [ ]:
# Checking missing values

df.isnull().sum()

In [ ]:
# Checking duplicate records

df.duplicated().sum()

In [ ]:
# Confirming data quality status

print("Missing values:", df.isnull().sum().sum())
print("Duplicate records:", df.duplicated().sum())

## 4. Exploratory Data Analysis

This section examines gender distribution, age distribution, annual income, and spending score patterns.

In [ ]:
# Customer distribution by gender

plt.figure(figsize=(6,4))
sns.countplot(x="Gender", data=df)
plt.title("Customer Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig("images/gender_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Age distribution

plt.figure(figsize=(8,5))
sns.histplot(df["Age"], bins=20)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("images/age_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Annual income distribution

plt.figure(figsize=(8,5))
sns.histplot(df["Annual Income (k$)"], bins=20)
plt.title("Annual Income Distribution")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("images/income_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Spending score distribution

plt.figure(figsize=(8,5))
sns.histplot(df["Spending Score (1-100)"], bins=20)
plt.title("Spending Score Distribution")
plt.xlabel("Spending Score")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("images/spending_score_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Feature Engineering

Age groups are created to support grouped customer analysis.

In [ ]:
# Creating customer age groups

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[18, 25, 35, 45, 55, 70],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"],
    include_lowest=True
)

# Checking age group distribution
df["Age_Group"].value_counts().sort_index()

In [ ]:
# Average spending score by gender

df.groupby("Gender")["Spending Score (1-100)"].mean().round(2)

In [ ]:
# Average annual income by gender

df.groupby("Gender")["Annual Income (k$)"].mean().round(2)

In [ ]:
# Average spending score by age group

df.groupby("Age_Group")["Spending Score (1-100)"].mean().round(2)

## 6. Correlation Analysis

Correlation analysis is used to examine relationships between numerical variables.

In [ ]:
# Correlation matrix

corr = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].corr()
corr

In [ ]:
# Correlation heatmap

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="Blues")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig("images/correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Annual income versus spending score

plt.figure(figsize=(8,6))
sns.scatterplot(
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    data=df
)
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score")
plt.tight_layout()
plt.savefig("images/income_vs_spending.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Preparing Data for Clustering

Annual income and spending score are selected because they represent customer value and purchasing behaviour.

In [ ]:
# Selecting variables for clustering

X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

# Display selected variables
X.head()

In [ ]:
# Standardising selected variables

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 8. Elbow Method

The Elbow Method is used to identify a suitable number of clusters for K-Means clustering.

In [ ]:
# Calculating WCSS values for different cluster numbers

wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

In [ ]:
# Plotting the Elbow Method

plt.figure(figsize=(8,5))
plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.tight_layout()
plt.savefig("images/elbow_method.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. K-Means Clustering

K-Means clustering is applied using five clusters based on the Elbow Method and common customer segmentation practice for this dataset.

In [ ]:
# Applying K-Means clustering

kmeans = KMeans(n_clusters=5, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Displaying updated dataset
df.head()

In [ ]:
# Number of customers in each cluster

df["Cluster"].value_counts().sort_index()

In [ ]:
# Cluster summary

cluster_summary = df.groupby("Cluster")[[
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)"
]].mean().round(2)

cluster_summary

In [ ]:
# Customer segmentation visualisation

plt.figure(figsize=(10,7))
sns.scatterplot(
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    data=df,
    palette="Set2"
)
plt.title("Customer Segmentation Using K-Means")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score")
plt.legend(title="Cluster")
plt.tight_layout()
plt.savefig("images/customer_segments.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Segment Labelling

Cluster labels are added to make the customer groups easier to interpret.

In [ ]:
# Creating business-friendly cluster labels

cluster_labels = {
    0: "Moderate Income - Moderate Spending",
    1: "High Income - Low Spending",
    2: "Low Income - High Spending",
    3: "Low Income - Low Spending",
    4: "High Income - High Spending"
}

df["Segment"] = df["Cluster"].map(cluster_labels)

df[["CustomerID", "Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)", "Cluster", "Segment"]].head()

In [ ]:
# Segment count

df["Segment"].value_counts()

In [ ]:
# Average values by segment

df.groupby("Segment")[[
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)"
]].mean().round(2)

## 11. Key Findings

The analysis grouped customers into five segments based on annual income and spending score. High-income and high-spending customers represent a valuable group for premium marketing strategies. High-income and low-spending customers may require targeted engagement. Low-income and high-spending customers show strong purchasing interest despite lower income levels. These findings can support targeted marketing and customer relationship strategies.

In [ ]:
# Saving the final dataset with clusters and segment labels

df.to_csv("mall_customers_with_segments.csv", index=False)

## 12. Optional Profiling Report

The profiling report can be used for a quick automated dataset summary. This cell may be commented before uploading to GitHub because the HTML output can increase notebook size.

In [ ]:
# Optional automated profiling report
# Run this cell only when needed.

# !pip install ydata-profiling -q
# from ydata_profiling import ProfileReport
# profile = ProfileReport(df, title="Mall Customers Profiling Report")
# profile.to_notebook_iframe()
# profile.to_file("mall_customers_profile_report.html")